# Gene2Wire BARseq M1: measurement degradation V3 — results-only 0914

This notebook combines three separately audited mechanisms: assay-specific gene
coverage, assay-specific target coverage, and assay×target positive censoring.
`GENE_COVERAGES` and `TARGET_COVERAGES` are **per-assay coverage**, not pairwise
overlap. At coverage 1.0, A, B and C each measure every available item. Thus BARseq
uses all 23 genes per assay at 1.0; the earlier fixed-eight-gene 100%-overlap endpoint
is not reused. Requested target coverage 0.40 is an explicit stress level, not a
bridge-complete design guarantee. Integer rounding and native availability in small-T
data (notably BARseq A1 with T=11 and MERGE-seq with T=5) can yield
`graph_connected=False`; the mask is not redrawn to force connectivity. Preflight and
exported support tables report the realized coverage and graph status.

Gene and target panels share the same nested, union-preserving cyclic generator, but
their downstream meanings differ. Missing genes are masked before train-only scaling
and enter the learner as value + observation-mask + assay-ID features. Missing targets
form `W_fit = W_native & panel`; those entries are omitted from every training loss and
are never labelled negative. Within measured target entries, heterogeneous censoring
hides positives only. The native reference test scope remains fixed across conditions.

Run top to bottom in a Python 3.10+ OnDemand kernel with the repository environment.
The notebook installs nothing, contains no local model/mask implementation, and uses
an exact source pin. Raw caches, checkpoints, complete CSV/NPZ exports, and PDF figures
persist below `GENE2WIRE_PROJECT_DIR`. `RESULTS_ONLY=True` redraws an explicit completed
export without loading raw data or launching fits. Canonical outputs are cleared.


This V3 notebook is a read-only visualization layer. It never trains, resumes checkpoints, or writes into the source export; experiment artifacts are written only below the separate V3 figure directory. The exact completed export is prefilled below.


## Run configuration and measurement grid


In [ ]:
from pathlib import Path
import os

# Shared scientific defaults. Change switches here before Run All.
PROTOCOL_VERSION = '0912-v6-measurement-degradation'
N_OUTER_FOLDS = 3
USE_LOCATION = False
USE_TARGET_FEATURES = False
N_JOBS = 32
PARALLEL_UNIT = 'scenario'
# Fast exploratory defaults. Increase both for formal uncertainty estimates.
N_REPETITIONS = 2  # Independent generated datasets per simulation sharing strength.
N_PANEL_SEEDS = 2  # Panel-mask repeats within each real or generated dataset.
# Explicit Joint search: rank screen -> top two ranks -> total/ratio refinement.
STRATEGY = 'rank_top2_total_ratio'
CANDIDATE_BUDGET = 32
SEED = 20260912
PAIRED_FRACTION = 0.20
CALIBRATION_FRACTIONS = (PAIRED_FRACTION,)

# Per-assay coverage: 1.0 means every assay measures every available item.
# These are coverage levels, not the legacy fixed-K pairwise-overlap levels.
GENE_COVERAGES = (1.0, 0.70, 0.40)
TARGET_COVERAGES = (1.0, 0.70, 0.40)
POSITIVE_RETENTIONS = (1.0, 0.70, 0.40, 0.10)
# Anchors must be members of their grids. Integer panel sizes can make realized
# coverage differ slightly; the preflight audit prints every realized value.
ANCHOR_GENE_COVERAGE = 0.70
ANCHOR_TARGET_COVERAGE = 0.70
ANCHOR_POSITIVE_RETENTION = 0.40
VIRTUAL_ASSAYS = ('A', 'B', 'C')
HETEROGENEITY_DELTA = 1.0
INCLUDE_MATCHED_UNIFORM_CONTROL = True
INCLUDE_NATURAL_RECOVERY = True

# Retain the complete shared benchmark/control suite and add the three PU comparators.
RUN_INFORMATION_CONTROLS = True
RUN_RANDOM_FOREST = True
RUN_MECHANISM_CONTROLS = True
RUN_CALIBRATION_CONTROLS = True
RUN_QIAO = True
RUN_PU_COMPARATORS = True

# Display switches never change the scientific cache identity.
SHOW_PROGRESS = True
PROGRESS_LEVEL = 'summary'
PROGRESS_INTERVAL_SECONDS = 60.0
SHOW_FULL_DIAGNOSTICS = False

BASE_DIR = Path(os.environ.get(
    'GENE2WIRE_PROJECT_DIR', '/home/yueyue/gene2wire')).expanduser()
RAW_DATA_DIR = BASE_DIR / 'raw_data'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints' / 'measurement_degradation_v2_compact'
EXPORT_DIR = BASE_DIR / 'paper_figure_exports'
FIGURE_DIR = BASE_DIR / 'figures' / 'measurement_degradation_v3' / '0914'
CODE_CACHE_DIR = BASE_DIR / 'code'

# Read-only redraw mode. Values are exact completed directories containing manifest.json.
RESULTS_ONLY = True
EXISTING_EXPORT_DIRS = {'BARseq M1': '/home/yueyue/gene2wire/paper_figure_exports/BARseq_M1_measurement_degradation/ad4abecbc92743f8d04f'}

SOURCE_RESULTS_CORE_COMMIT = '5fce6cab3c7662256ad1f82d0576188e00389d07'
EXPECTED_RESULTS_SOURCE_HASH = '45011b9f2b618c53703a02fe6dcdcac70250b793caf9231299ef5e8735b39eaa'
EXPECTED_RESULTS_RUN_ID = 'ad4abecbc92743f8d04f'
V3_RESULTS_POLICY = 'read-only exact export; never train or mutate prior results'

CORE_COMMIT = 'c1793c386e096b48b91513f244ef6fc24e243025'
EXPECTED_SOURCE_HASH = 'abccf031027c0e97ef6d1a8273afa3bb2b2b10fd6cd18f0cc0fcd6dbfcfc7f14'
REPO_URL = 'https://github.com/Yue-stat/Gene2Wire.git'

# The outer pool is the sole source of parallelism.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REQUIRED_MODULES = ['numpy', 'scipy', 'pandas', 'sklearn', 'joblib', 'threadpoolctl', 'matplotlib', 'yaml', 'IPython']
EXPECTED_EXPORT_LABELS = ('BARseq M1',)

if RESULTS_ONLY is not True:
    raise RuntimeError('V3 is results-only. Do not use it to fit or resume models.')


## Load and verify the frozen shared core

A matching local checkout is accepted without network access. Otherwise the exact
commit is cached and its byte-level source hash is verified before import.


In [ ]:
import hashlib
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile

if sys.version_info < (3, 10):
    raise RuntimeError('Select an OnDemand Python 3.10 or newer kernel.')
if not re.fullmatch(r'[0-9a-f]{40}', CORE_COMMIT):
    raise RuntimeError('This notebook needs its released 40-character CORE_COMMIT pin.')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_SOURCE_HASH):
    raise RuntimeError('This notebook needs its released source checksum.')

def notebook_source_hash(package_root):
    # Same byte-level convention as experiments.protocol.source_hash().
    digest = hashlib.sha256()
    for source_path in sorted(package_root.rglob('*.py')):
        digest.update(source_path.relative_to(package_root).as_posix().encode())
        digest.update(source_path.read_bytes())
    return digest.hexdigest()

def verify_checkout(checkout, require_git_pin=True):
    package_root = checkout / 'src' / 'gene2wire'
    if not package_root.is_dir():
        raise RuntimeError(f'Missing Gene2Wire sources in {checkout}')
    if require_git_pin:
        actual_commit = subprocess.check_output(
            ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        if actual_commit != CORE_COMMIT:
            raise RuntimeError(f'Cached code has commit {actual_commit}, expected {CORE_COMMIT}.')
    if notebook_source_hash(package_root) != EXPECTED_SOURCE_HASH:
        raise RuntimeError(f'Source checksum mismatch in {checkout}; use the released code.')
    return checkout

# Running from the exact local repository is supported without any network call.
CORE_CHECKOUT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    package_root = candidate / 'src' / 'gene2wire'
    if package_root.is_dir() and notebook_source_hash(package_root) == EXPECTED_SOURCE_HASH:
        CORE_CHECKOUT = verify_checkout(candidate, require_git_pin=False)
        break

if CORE_CHECKOUT is None:
    CODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached_checkout = CODE_CACHE_DIR / CORE_COMMIT
    if not cached_checkout.exists():
        stage = Path(tempfile.mkdtemp(prefix='.gene2wire-download-', dir=CODE_CACHE_DIR))
        try:
            for arguments in (
                ['git', 'init', '--quiet', str(stage)],
                ['git', '-C', str(stage), 'remote', 'add', 'origin', REPO_URL],
                ['git', '-C', str(stage), 'fetch', '--quiet', '--depth', '1', 'origin', CORE_COMMIT],
                ['git', '-C', str(stage), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'],
            ):
                subprocess.run(arguments, check=True)
            verify_checkout(stage)
            try:
                stage.rename(cached_checkout)
            except OSError:
                # Another notebook may have completed this same immutable cache.
                if not cached_checkout.exists():
                    raise
                verify_checkout(cached_checkout)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
    CORE_CHECKOUT = verify_checkout(cached_checkout)

existing = sys.modules.get('gene2wire')
if existing is not None:
    same_path = Path(existing.__file__).resolve().parent == (CORE_CHECKOUT / 'src' / 'gene2wire').resolve()
    same_source = getattr(existing, '_notebook_source_hash_0908', None) == EXPECTED_SOURCE_HASH
    if not (same_path and same_source):
        raise RuntimeError('A different or unverified gene2wire is already imported. Restart the kernel, then Run All.')

missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Use an OnDemand Python kernel containing these dependencies: '
                       + ', '.join(missing) + '. See the repository environment instructions.')
sys.path.insert(0, str(CORE_CHECKOUT / 'src'))
import gene2wire
from gene2wire.experiments.protocol import Settings, source_hash
if source_hash() != EXPECTED_SOURCE_HASH:
    raise RuntimeError('Imported code does not match the released source checksum.')
gene2wire._notebook_source_hash_0908 = EXPECTED_SOURCE_HASH
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print({'core_commit': CORE_COMMIT, 'source_hash': source_hash(),
       'imported_from': gene2wire.__file__,
       'v3_figure_dir': str(FIGURE_DIR), 'results_only': True})


## Shared models, comparators and controls

PU-Joint and the full retained benchmark suite use the same split, observed masks,
paired-reference budget and candidate ceiling. This family additionally enables
GenEML-adapted, Inductive-PU-MC and SAR-PU. The `-adapted` label is retained until the
Python 3 port is validated as implementation-equivalent to the historical GenEML code.
`rank_top2_total_ratio` explicitly screens declared Joint ranks, retains the two best
distinct converged ranks using development validation, and allocates the remaining
native budget across total-shrinkage and residual/shared-ratio coordinates.


In [ ]:
from dataclasses import asdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from gene2wire.experiments.measurement_experiment import MeasurementConfig
from gene2wire.experiments.measurement_plotting import (
    plot_gene_target_brier_heatmap,
)
from gene2wire.experiments.reporting import (
    configure_compact_display,
    configure_full_display,
    display_diagnostics,
    load_existing_exports,
)

if SHOW_FULL_DIAGNOSTICS:
    configure_full_display()
else:
    configure_compact_display()

settings = Settings(
    protocol_version=PROTOCOL_VERSION,
    n_outer_folds=N_OUTER_FOLDS,
    use_location=USE_LOCATION,
    use_target_features=USE_TARGET_FEATURES,
    n_jobs=N_JOBS,
    parallel_unit=PARALLEL_UNIT,
    n_repetitions=N_REPETITIONS,
    strategy=STRATEGY,
    seed=SEED,
    paired_fraction=PAIRED_FRACTION,
    calibration_fractions=CALIBRATION_FRACTIONS,
    loss_rates=tuple(1.0 - value for value in POSITIVE_RETENTIONS),
    candidate_budget=CANDIDATE_BUDGET,
    run_information_controls=RUN_INFORMATION_CONTROLS,
    run_random_forest=RUN_RANDOM_FOREST,
    run_mechanism_controls=RUN_MECHANISM_CONTROLS,
    run_calibration_controls=RUN_CALIBRATION_CONTROLS,
    run_qiao=RUN_QIAO,
    run_pu_comparators=RUN_PU_COMPARATORS,
    supervision_profile='paired_reference',
)
measurement_config = MeasurementConfig(
    gene_coverages=GENE_COVERAGES,
    target_coverages=TARGET_COVERAGES,
    retentions=POSITIVE_RETENTIONS,
    anchor_gene_coverage=ANCHOR_GENE_COVERAGE,
    anchor_target_coverage=ANCHOR_TARGET_COVERAGE,
    anchor_retention=ANCHOR_POSITIVE_RETENTION,
    assay_ids=VIRTUAL_ASSAYS,
    heterogeneity_delta=HETEROGENEITY_DELTA,
    include_matched_uniform=INCLUDE_MATCHED_UNIFORM_CONTROL,
    include_natural_recovery=INCLUDE_NATURAL_RECOVERY,
    n_panel_seeds=N_PANEL_SEEDS,
)
KEY_MEASUREMENT_MODELS = (
    'PU', 'PU-MIRT', 'PU-Joint',
    'GenEML-adapted', 'Inductive-PU-MC', 'SAR-PU',
)

if SHOW_FULL_DIAGNOSTICS:
    display(pd.DataFrame([asdict(settings)]).T.rename(columns={0: 'shared setting'}))
    display(pd.DataFrame([asdict(measurement_config)]).T.rename(columns={0: 'measurement design'}))
else:
    print({
        'folds': N_OUTER_FOLDS, 'repetitions': N_REPETITIONS,
        'panel_seeds_per_dataset': N_PANEL_SEEDS,
        'n_jobs': N_JOBS, 'parallel_unit': PARALLEL_UNIT,
        'use_location': USE_LOCATION, 'use_target_features': USE_TARGET_FEATURES,
        'strategy': STRATEGY, 'candidate_budget': CANDIDATE_BUDGET,
        'paired_fraction': PAIRED_FRACTION,
        'gene_coverages': GENE_COVERAGES,
        'target_coverages': TARGET_COVERAGES,
        'positive_retentions': POSITIVE_RETENTIONS,
        'virtual_assays': VIRTUAL_ASSAYS,
        'all_controls': {
            'information': RUN_INFORMATION_CONTROLS,
            'random_forest': RUN_RANDOM_FOREST,
            'mechanism': RUN_MECHANISM_CONTROLS,
            'calibration': RUN_CALIBRATION_CONTROLS,
            'qiao': RUN_QIAO,
            'pu_comparators': RUN_PU_COMPARATORS,
        },
    })

if RESULTS_ONLY:
    all_artifacts = load_existing_exports(
        EXISTING_EXPORT_DIRS, expected_labels=EXPECTED_EXPORT_LABELS)
    print('RESULTS_ONLY: raw loading, design preview and fitting are skipped.')
    print('Plots and diagnostics use each export manifest as the authoritative settings record.')

for label, artifacts in all_artifacts.items():
    manifest = getattr(artifacts, 'manifest', {}) or {}
    if manifest.get('experiment') != 'measurement_degradation':
        raise RuntimeError(
            f'{label}: expected a measurement_degradation export, got '
            f'{manifest.get("experiment")!r}.')
    if manifest.get('completed') is not True:
        raise RuntimeError(f'{label}: source export is not marked completed.')
    if (EXPECTED_RESULTS_RUN_ID is not None
            and str(manifest.get('run_id')) != EXPECTED_RESULTS_RUN_ID):
        raise RuntimeError(
            f'{label}: export run ID {manifest.get("run_id")!r} does not match '
            f'the pinned V3 source result {EXPECTED_RESULTS_RUN_ID!r}.')
    if (EXPECTED_RESULTS_SOURCE_HASH is not None
            and manifest.get('source_hash') != EXPECTED_RESULTS_SOURCE_HASH):
        raise RuntimeError(
            f'{label}: source export hash {manifest.get("source_hash")!r} does not '
            f'match the pinned result hash {EXPECTED_RESULTS_SOURCE_HASH!r}.')
    saved_version = (manifest.get('protocol') or {}).get('protocol_version')
    if saved_version != PROTOCOL_VERSION:
        raise RuntimeError(
            f'{label}: export protocol {saved_version!r} does not match '
            f'{PROTOCOL_VERSION!r}.')
    saved_measurement = manifest.get('measurement_protocol') or {}
    expected_measurement = {
        'gene_coverages': GENE_COVERAGES,
        'target_coverages': TARGET_COVERAGES,
        'retentions': POSITIVE_RETENTIONS,
    }
    for coordinate, expected_values in expected_measurement.items():
        saved_values = saved_measurement.get(coordinate)
        if (saved_values is None or len(saved_values) != len(expected_values)
                or not np.allclose(
                    np.asarray(saved_values, dtype=float),
                    np.asarray(expected_values, dtype=float))):
            raise RuntimeError(
                f'{label}: export {coordinate}={saved_values!r} does not match '
                f'the V3 result contract {expected_values!r}.')
    expected_anchors = {
        'anchor_gene_coverage': ANCHOR_GENE_COVERAGE,
        'anchor_target_coverage': ANCHOR_TARGET_COVERAGE,
        'anchor_retention': ANCHOR_POSITIVE_RETENTION,
        'n_panel_seeds': N_PANEL_SEEDS,
    }
    for coordinate, expected_value in expected_anchors.items():
        saved_value = saved_measurement.get(coordinate)
        if saved_value is None or not np.isclose(float(saved_value), float(expected_value)):
            raise RuntimeError(
                f'{label}: export {coordinate}={saved_value!r} does not match '
                f'the V3 result contract {expected_value!r}.')
    if str(artifacts.export_dir.name) != str(manifest.get('run_id')):
        raise RuntimeError(
            f'{label}: export directory run ID and manifest run ID disagree.')
    required_tables = {'metrics', 'per_repetition', 'heatmap_baseline_selection'}
    if label == 'Projection-TAGs':
        required_tables.update({'projection_budget_recall',
                                'projection_budget_recall_per_target'})
    missing_tables = sorted(required_tables.difference(artifacts.tables))
    if missing_tables:
        raise RuntimeError(f'{label}: completed export lacks tables {missing_tables}.')
    print({
        'results_only_dataset': label,
        'read_only_export': str(artifacts.export_dir),
        'run_id': manifest.get('run_id'),
        'source_result_core_commit': SOURCE_RESULTS_CORE_COMMIT,
        'export_source_hash': manifest.get('source_hash'),
        'export_protocol_version': (manifest.get('protocol') or {}).get('protocol_version'),
        'v3_writes_only_to': str(FIGURE_DIR),
    })


## CPU allocation and live workers

The worker display distinguishes requested/effective experiment workers from detected
machine capacity. Numerical libraries remain single-threaded inside each outer worker.


In [ ]:
from gene2wire.experiments.workers import NotebookWorkerStatus

worker_status = None
if RESULTS_ONLY:
    print('RESULTS_ONLY: no training workers are launched.')
else:
    worker_status = NotebookWorkerStatus(requested_workers=N_JOBS)


## Figure helpers and fixed evaluation scope


In [ ]:
def _role_rows(frame, role):
    required = {'condition_roles', 'evaluation_scope', 'mechanism'}
    missing = required.difference(frame.columns)
    if missing:
        raise RuntimeError(f'metrics.csv lacks measurement coordinates: {sorted(missing)}')
    return frame.loc[
        frame['condition_roles'].astype(str).str.contains(role, regex=False)
        & frame['evaluation_scope'].eq('native_reference')
        & frame['mechanism'].eq('assay_target_sar')
    ].copy()


def _plot_groups(frame):
    if 'sharing_strength' not in frame or frame['sharing_strength'].isna().all():
        return [('all', frame)]
    return [(f'sharing_{value:g}', group) for value, group in frame.groupby(
        'sharing_strength', dropna=False, observed=True)]


def _figure_path(label, suffix):
    safe = ''.join(character if character.isalnum() or character in '-_' else '_'
                   for character in str(label)).strip('_')
    path = FIGURE_DIR / f'{safe}_{suffix}.pdf'
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _save_show(figure, label, suffix):
    path = _figure_path(label, suffix)
    figure.savefig(path, bbox_inches='tight')
    display(path)
    plt.show()
    plt.close(figure)
    return path

"""Results-only analysis helpers embedded in measurement-degradation V3 notebooks.

This module contains plotting and evaluation code only.  It never fits a model,
changes a checkpoint, or writes into a completed experiment export.  The V3
builder embeds :func:`notebook_source` into each generated notebook so the
notebooks remain self-contained when opened outside a repository checkout.
"""
from collections.abc import Mapping, Sequence
import inspect
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from gene2wire.experiments.measurement_plotting import (
    FULL_MODEL_ORDER,
    KEY_MODELS,
    MODEL_COLORS,
)
from gene2wire.experiments.pipeline import slug


FUNKY_METRICS = (
    ("macro_auprc", "AUPRC", "circle", True),
    ("macro_log_loss", "Log loss", "bar", False),
    ("macro_auroc", "AUROC", "circle", True),
    ("macro_brier", "Brier", "bar", False),
)

REFERENCE_PROBABILITY_PREFIX = "reference"
EXTRA_INFORMATION_MODELS = frozenset({
    "Reference-only",
    "Reference+PU",
    "Reference+PU-MIRT",
    "Reference+PU-Joint",
    "RF-reference",
    "RF-mixed",
})


def _as_frame(value, *, name: str, required: Sequence[str] = ()) -> pd.DataFrame:
    if not isinstance(value, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame")
    missing = set(required).difference(value.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")
    return value.copy()


def _numeric(frame: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(frame[column], errors="coerce")


def _roles(frame: pd.DataFrame, role: str) -> pd.Series:
    return frame["condition_roles"].astype(str).str.contains(role, regex=False)


def _reference_probability(frame: pd.DataFrame) -> pd.Series:
    if "probability_semantics" not in frame:
        return pd.Series(True, index=frame.index, dtype=bool)
    return frame["probability_semantics"].fillna("").astype(str).str.startswith(
        REFERENCE_PROBABILITY_PREFIX
    )


def _ordered_models(values: Sequence[str], *, key_models=KEY_MODELS) -> list[str]:
    present = list(dict.fromkeys(map(str, values)))
    priority = {model: index for index, model in enumerate(FULL_MODEL_ORDER)}
    key_priority = {model: index for index, model in enumerate(key_models)}
    return sorted(
        present,
        key=lambda model: (
            0 if model in key_priority else 1,
            key_priority.get(model, priority.get(model, len(priority))),
            model,
        ),
    )


def _models_for_mode(frame: pd.DataFrame, mode: str, key_models=KEY_MODELS) -> list[str]:
    if mode not in {"key", "full"}:
        raise ValueError("mode must be 'key' or 'full'")
    present = set(frame["model"].dropna().astype(str))
    if mode == "key":
        # Keep the declared rows even when a model failed every condition.  A
        # fully absent model must appear as NA instead of silently vanishing.
        models = list(dict.fromkeys(map(str, key_models)))
    else:
        extras = sorted(present.difference(FULL_MODEL_ORDER))
        models = [*FULL_MODEL_ORDER, *extras]
    if not models:
        raise ValueError(f"No {mode} models are present in the plotting table")
    return models


def _model_color(model: str, index: int):
    if model in MODEL_COLORS:
        return MODEL_COLORS[model]
    palette = plt.get_cmap("tab20")
    return palette(index % palette.N)


def plot_degradation_pair(
    table: pd.DataFrame,
    *,
    x_col: str,
    x_label: str,
    mode: str = "key",
    key_models: Sequence[str] = KEY_MODELS,
    title: str | None = None,
) -> tuple[Figure, np.ndarray]:
    """Plot Macro-AUPRC and reference-probability log loss side by side.

    The degradation coordinate always runs from 100% at the left toward 0% at
    the right.  Ranking metrics remain available for every model.  The proper
    log-loss panel omits observed/mixed-probability models rather than treating
    a detection probability as a reference probability.
    """
    frame = _as_frame(
        table,
        name="degradation table",
        required=(x_col, "model", "macro_auprc", "macro_log_loss"),
    )
    if frame.empty:
        raise ValueError("degradation table must contain at least one row")
    frame[x_col] = _numeric(frame, x_col)
    frame["macro_auprc"] = _numeric(frame, "macro_auprc")
    frame["macro_log_loss"] = _numeric(frame, "macro_log_loss")
    frame = frame.loc[np.isfinite(frame[x_col]) & frame[x_col].between(0, 1)]
    models = _models_for_mode(frame, mode, key_models)
    figure, axes = plt.subplots(1, 2, figsize=(13.2, 4.5), constrained_layout=True)
    specifications = (
        ("macro_auprc", "Macro AUPRC ↑", pd.Series(True, index=frame.index)),
        ("macro_log_loss", "Macro log loss ↓", _reference_probability(frame)),
    )
    for axis, (metric, ylabel, semantic_mask) in zip(axes, specifications):
        plotting = frame.loc[semantic_mask & np.isfinite(frame[metric])].copy()
        summary = (
            plotting.groupby(["model", x_col], observed=True, sort=False)[metric]
            .mean()
            .reset_index()
        )
        for index, model in enumerate(models):
            selected = summary.loc[summary["model"].astype(str).eq(model)].sort_values(
                x_col, ascending=False
            )
            if selected.empty:
                continue
            axis.plot(
                selected[x_col],
                selected[metric],
                marker="o",
                linewidth=1.8,
                markersize=4.5,
                label=model,
                color=_model_color(model, index),
            )
        axis.set_xlim(1.0, 0.0)
        axis.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        axis.set_xlabel(f"{x_label} (100% → 0%)")
        axis.set_ylabel(ylabel)
        axis.grid(axis="y", color="#E6E6E6", linewidth=0.7)
        axis.spines[["top", "right"]].set_visible(False)
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        figure.legend(
            handles,
            labels,
            frameon=False,
            loc="outside lower center",
            ncol=min(4, max(1, len(labels))),
        )
    figure.suptitle(title or f"{x_label} degradation — {mode} models")
    return figure, axes


def augment_projection_budget_metrics(
    table: pd.DataFrame,
    *,
    per_target: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add confirmation yield and union-reference false-positive measures.

    Macro ratios are computed from the per-target table.  Pooled ratios are
    explicitly named ``micro_*``.  A union-negative selection is an assay-
    reference false positive, not proof of a biological false projection.
    """
    required = (
        "dataset",
        "repetition",
        "model",
        "budget_fraction",
        "selected_candidates",
        "candidate_count",
        "amplification_confirmed_positive_count",
        "recovered_positive_count",
    )
    summary = _as_frame(table, name="Projection budget table", required=required)
    targets = _as_frame(
        pd.DataFrame() if per_target is None else per_target,
        name="Projection per-target budget table",
    )
    for frame in (summary, targets):
        if frame.empty:
            continue
        missing = set(required).difference(frame.columns)
        if missing:
            raise ValueError(
                "Projection per-target budget table is missing required columns: "
                f"{sorted(missing)}"
            )
        for column in (
            "budget_fraction",
            "selected_candidates",
            "candidate_count",
            "amplification_confirmed_positive_count",
            "recovered_positive_count",
        ):
            frame[column] = _numeric(frame, column)
        frame["amplification_reference_false_positive_count"] = (
            frame["selected_candidates"] - frame["recovered_positive_count"]
        )
        frame["amplification_reference_negative_count"] = (
            frame["candidate_count"]
            - frame["amplification_confirmed_positive_count"]
        )
        frame["amplification_confirmed_false_negative_count"] = (
            frame["amplification_confirmed_positive_count"]
            - frame["recovered_positive_count"]
        )
        frame["amplification_reference_true_negative_count"] = (
            frame["amplification_reference_negative_count"]
            - frame["amplification_reference_false_positive_count"]
        )
        selected = frame["selected_candidates"].to_numpy(dtype=float)
        negatives = frame["amplification_reference_negative_count"].to_numpy(dtype=float)
        candidates = frame["candidate_count"].to_numpy(dtype=float)
        positives = frame["amplification_confirmed_positive_count"].to_numpy(dtype=float)
        recovered = frame["recovered_positive_count"].to_numpy(dtype=float)
        false_positive = frame[
            "amplification_reference_false_positive_count"
        ].to_numpy(dtype=float)
        precision = np.divide(
            recovered,
            selected,
            out=np.full_like(recovered, np.nan),
            where=selected > 0,
        )
        recall = np.divide(
            recovered,
            positives,
            out=np.full_like(recovered, np.nan),
            where=positives > 0,
        )
        prevalence = np.divide(
            positives,
            candidates,
            out=np.full_like(recovered, np.nan),
            where=candidates > 0,
        )
        frame["amplification_confirmed_precision"] = precision
        frame["amplification_confirmed_recall"] = recall
        frame["amplification_reference_fdr"] = 1.0 - precision
        frame["amplification_reference_fpr"] = np.divide(
            false_positive,
            negatives,
            out=np.full_like(recovered, np.nan),
            where=negatives > 0,
        )
        frame["amplification_confirmed_f1"] = np.divide(
            2 * precision * recall,
            precision + recall,
            out=np.full_like(recovered, np.nan),
            where=np.isfinite(precision + recall) & ((precision + recall) > 0),
        )
        frame["amplification_confirmed_lift"] = np.divide(
            precision,
            prevalence,
            out=np.full_like(recovered, np.nan),
            where=prevalence > 0,
        )

    if not targets.empty:
        keys = ["dataset", "repetition", "model", "budget_fraction"]
        ratios = (
            "amplification_confirmed_recall",
            "amplification_confirmed_precision",
            "amplification_reference_fdr",
            "amplification_reference_fpr",
            "amplification_confirmed_f1",
            "amplification_confirmed_lift",
        )
        macro = targets.groupby(keys, dropna=False, observed=True)[list(ratios)].mean()
        summary = summary.drop(columns=[name for name in ratios if name in summary], errors="ignore")
        summary = summary.merge(macro.reset_index(), on=keys, how="left", validate="one_to_one")

    # Aggregate-count ratios are always retained with a micro prefix.
    selected = summary["selected_candidates"].to_numpy(dtype=float)
    candidates = summary["candidate_count"].to_numpy(dtype=float)
    positives = summary["amplification_confirmed_positive_count"].to_numpy(dtype=float)
    recovered = summary["recovered_positive_count"].to_numpy(dtype=float)
    negatives = candidates - positives
    false_positive = selected - recovered
    micro_precision = np.divide(
        recovered, selected, out=np.full_like(recovered, np.nan), where=selected > 0
    )
    micro_recall = np.divide(
        recovered, positives, out=np.full_like(recovered, np.nan), where=positives > 0
    )
    prevalence = np.divide(
        positives, candidates, out=np.full_like(recovered, np.nan), where=candidates > 0
    )
    summary["micro_amplification_confirmed_precision"] = micro_precision
    summary["micro_amplification_confirmed_recall"] = micro_recall
    summary["micro_amplification_reference_fdr"] = 1.0 - micro_precision
    summary["micro_amplification_reference_fpr"] = np.divide(
        false_positive,
        negatives,
        out=np.full_like(recovered, np.nan),
        where=negatives > 0,
    )
    summary["micro_amplification_confirmed_f1"] = np.divide(
        2 * micro_precision * micro_recall,
        micro_precision + micro_recall,
        out=np.full_like(recovered, np.nan),
        where=np.isfinite(micro_precision + micro_recall)
        & ((micro_precision + micro_recall) > 0),
    )
    summary["micro_amplification_confirmed_lift"] = np.divide(
        micro_precision,
        prevalence,
        out=np.full_like(recovered, np.nan),
        where=prevalence > 0,
    )
    return summary, targets


def plot_projection_budget_metrics(
    table: pd.DataFrame,
    *,
    models: Sequence[str] = KEY_MODELS,
    title: str | None = None,
) -> tuple[Figure, np.ndarray]:
    """Plot amplification-confirmed recovery and reference FP trade-offs."""
    frame = _as_frame(
        table,
        name="Projection budget metrics",
        required=(
            "model",
            "budget_fraction",
            "amplification_confirmed_recall",
            "amplification_confirmed_precision",
            "amplification_reference_fdr",
            "amplification_reference_fpr",
            "amplification_confirmed_f1",
            "amplification_confirmed_lift",
        ),
    )
    order = [model for model in models if model in set(frame["model"].astype(str))]
    if not order:
        raise ValueError("Projection budget metrics contain none of the requested models")
    specifications = (
        ("amplification_confirmed_recall", "Confirmed recall ↑", True),
        ("amplification_confirmed_precision", "Confirmation precision ↑", True),
        ("amplification_confirmed_f1", "Confirmed F1 ↑", True),
        ("amplification_confirmed_lift", "Enrichment over random ↑", False),
        ("amplification_reference_fdr", "Union-reference FDR ↓", True),
        ("amplification_reference_fpr", "Union-reference FPR ↓", True),
    )
    figure, axes = plt.subplots(2, 3, figsize=(15.5, 8.0), constrained_layout=True)
    for axis, (metric, ylabel, percentage) in zip(axes.flat, specifications):
        plotting = frame.copy()
        plotting[metric] = _numeric(plotting, metric)
        summary = (
            plotting.groupby(["model", "budget_fraction"], observed=True)[metric]
            .mean()
            .reset_index()
        )
        for index, model in enumerate(order):
            selected = summary.loc[summary["model"].astype(str).eq(model)].sort_values(
                "budget_fraction"
            )
            axis.plot(
                selected["budget_fraction"],
                selected[metric],
                marker="o",
                linewidth=1.7,
                markersize=4,
                label=model,
                color=_model_color(model, index),
            )
        axis.set_xlabel("Top-ranked candidate budget")
        axis.set_ylabel(ylabel)
        axis.set_xlim(left=0.0)
        axis.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        if percentage:
            axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        axis.grid(axis="y", color="#E6E6E6", linewidth=0.7)
        axis.spines[["top", "right"]].set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        frameon=False,
        loc="outside lower center",
        ncol=min(4, len(labels)),
    )
    figure.suptitle(title or "Projection-TAGs amplification-reference recovery")
    return figure, axes


def _metric_vector(truth: np.ndarray, probability: np.ndarray | None,
                   ranking: np.ndarray) -> dict[str, float]:
    truth = np.asarray(truth, dtype=int)
    ranking = np.asarray(ranking, dtype=float)
    both = len(truth) > 0 and np.unique(truth).size == 2
    result = {
        "auprc": float(average_precision_score(truth, ranking)) if both else np.nan,
        "auroc": float(roc_auc_score(truth, ranking)) if both else np.nan,
        "brier": np.nan,
        "log_loss": np.nan,
        "predicted_prevalence": np.nan,
        "reference_prevalence": float(np.mean(truth)) if len(truth) else np.nan,
    }
    if probability is not None:
        values = np.asarray(probability, dtype=float)
        if values.shape != truth.shape or not np.all(np.isfinite(values)):
            raise ValueError("Candidate probabilities must be finite and aligned")
        values = np.clip(values, 1e-9, 1 - 1e-9)
        result.update(
            brier=float(np.mean((truth - values) ** 2)),
            log_loss=float(np.mean(-truth * np.log(values) - (1 - truth) * np.log1p(-values))),
            predicted_prevalence=float(np.mean(values)),
        )
    return result


def _natural_semantics(metrics: pd.DataFrame) -> dict[str, str]:
    required = {"model", "mechanism", "condition_roles", "probability_semantics"}
    if not required.issubset(metrics):
        raise ValueError(
            "metrics.csv must record model probability_semantics for natural recovery"
        )
    frame = metrics.loc[
        metrics["mechanism"].astype(str).eq("natural")
        & _roles(metrics, "natural_recovery")
    ]
    result = {}
    for model, values in frame.groupby("model", observed=True)["probability_semantics"]:
        unique = tuple(sorted(set(values.dropna().astype(str))))
        if len(unique) != 1:
            raise ValueError(
                f"Natural recovery has ambiguous probability semantics for {model}: {unique}"
            )
        result[str(model)] = unique[0]
    return result


def _posterior_after_nondetection(reference_probability, sensitivity):
    p = np.asarray(reference_probability, dtype=float)
    e = np.asarray(sensitivity, dtype=float)
    if p.shape != e.shape or not np.all(np.isfinite(p)) or not np.all(np.isfinite(e)):
        raise ValueError("Reference probability and sensitivity must be finite and aligned")
    if np.any((p < 0) | (p > 1)) or np.any((e < 0) | (e > 1)):
        raise ValueError("Reference probability and sensitivity must lie in [0, 1]")
    denominator = 1.0 - e * p
    return np.divide(
        (1.0 - e) * p,
        denominator,
        out=np.full_like(p, np.nan),
        where=denominator > 1e-12,
    )


def derive_projection_natural_metrics(
    artifacts,
    *,
    budgets: Sequence[float] = (0.01, 0.05, 0.10, 0.20),
) -> dict[str, pd.DataFrame]:
    """Re-evaluate saved natural OOF candidates without fitting or export writes.

    For a reference-probability model, standard non-detection changes the
    candidate probability from ``p`` to ``h=(1-e)p/(1-ep)``.  Ranking-only
    observed/mixed models retain their saved score but receive no Brier or log
    loss.  The returned tables are newly allocated in memory.
    """
    metrics = _as_frame(
        artifacts.tables.get("metrics", pd.DataFrame()),
        name="metrics.csv",
        required=("model", "mechanism", "condition_roles", "probability_semantics"),
    )
    semantics = _natural_semantics(metrics)
    model_lookup = {slug(model): model for model in semantics}
    export_dir = Path(artifacts.export_dir)
    units = export_dir / "units"
    if not units.is_dir():
        raise FileNotFoundError(
            f"Projection natural re-evaluation needs saved OOF units below {units}"
        )
    budgets = tuple(float(value) for value in budgets)
    if not budgets or len(set(budgets)) != len(budgets) or any(
        not np.isfinite(value) or value <= 0 or value > 1 for value in budgets
    ):
        raise ValueError("budgets must contain distinct fractions in (0, 1]")

    manifest = getattr(artifacts, "manifest", {}) or {}
    measurement_protocol = manifest.get("measurement_protocol") or {}
    saved_protocol = manifest.get("protocol") or {}
    try:
        expected_panel_seed_count = int(measurement_protocol["n_panel_seeds"])
        expected_outer_fold_count = int(saved_protocol["n_outer_folds"])
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError(
            "Projection natural re-evaluation needs manifest n_panel_seeds and "
            "n_outer_folds to prove OOF completeness"
        ) from error
    if expected_panel_seed_count < 1 or expected_outer_fold_count < 1:
        raise ValueError("Manifest panel-seed and outer-fold counts must be positive")
    expected_seed_folds = {
        (panel_seed, outer_fold)
        for panel_seed in range(expected_panel_seed_count)
        for outer_fold in range(expected_outer_fold_count)
    }

    print(
        f"[V3 Projection] discover natural-recovery OOF files below {units}",
        flush=True,
    )
    expected_contexts: dict[tuple[str, int], set[tuple[int, int]]] = {}
    prediction_paths: dict[tuple[str, int, str, int, int], Path] = {}
    for audit_path in sorted(units.glob("*/audit.json")):
        context = json.loads(audit_path.read_text(encoding="utf-8"))
        if context.get("mechanism") != "natural" or "natural_recovery" not in str(
            context.get("condition_roles", "")
        ):
            continue
        dataset = str(context.get("dataset"))
        # Real-data ``repetition`` is the panel seed; ``data_repetition`` is the
        # independent data unit. Natural full-panel copies are collapsed only
        # after their arrays are verified identical.
        repetition = int(context.get(
            "data_repetition", context.get("repetition", 0)
        ))
        panel_seed = int(context.get("panel_seed", context.get("repetition", 0)))
        outer_fold = int(context.get("outer_fold", -1))
        expected_contexts.setdefault((dataset, repetition), set()).add(
            (panel_seed, outer_fold)
        )
        for prediction_path in sorted(audit_path.parent.glob("*_predictions.npz")):
            model = model_lookup.get(prediction_path.name[: -len("_predictions.npz")])
            if model is None:
                continue
            key = (dataset, repetition, model, panel_seed, outer_fold)
            if key in prediction_paths:
                raise ValueError(
                    f"Duplicate natural prediction file for {key}: "
                    f"{prediction_paths[key]} and {prediction_path}"
                )
            prediction_paths[key] = prediction_path
    if not prediction_paths:
        raise ValueError("No saved Projection-TAGs natural-recovery OOF candidates were found")
    print(
        f"[V3 Projection] found {len(prediction_paths)} prediction files across "
        f"{len(expected_contexts)} data-repetition context(s) and "
        f"{len(semantics)} model(s)",
        flush=True,
    )
    for (dataset, repetition), contexts in sorted(expected_contexts.items()):
        if contexts != expected_seed_folds:
            raise ValueError(
                "Natural-recovery OOF audit contexts are incomplete for "
                f"{dataset}, data repetition={repetition}; "
                f"missing={sorted(expected_seed_folds.difference(contexts))}, "
                f"extra={sorted(contexts.difference(expected_seed_folds))}"
            )

    def load_candidate_arrays(path: Path, model: str) -> dict[str, object]:
        with np.load(path, allow_pickle=False) as archive:
            arrays = {name: archive[name].copy() for name in archive.files}
        required_arrays = {
            "prediction", "reference", "observed", "source_measured",
            "cell_ids", "target_ids",
        }
        missing = required_arrays.difference(arrays)
        if missing:
            raise ValueError(
                f"{path} lacks natural-recovery arrays: {sorted(missing)}"
            )
        prediction = np.asarray(arrays["prediction"], dtype=float)
        reference = np.asarray(arrays["reference"], dtype=bool)
        observed = np.asarray(arrays["observed"], dtype=bool)
        measured = np.asarray(arrays["source_measured"], dtype=bool)
        if not (
            prediction.shape == reference.shape == observed.shape == measured.shape
        ):
            raise ValueError(f"Natural-recovery arrays have different shapes in {path}")
        cells = np.asarray(arrays["cell_ids"]).astype(str)
        targets = np.asarray(arrays["target_ids"]).astype(str)
        if cells.shape != (len(prediction),) or targets.shape != (prediction.shape[1],):
            raise ValueError(f"Natural-recovery IDs do not align in {path}")
        semantics_value = semantics[model]
        if semantics_value.startswith(REFERENCE_PROBABILITY_PREFIX):
            if "estimated_sensitivity" not in arrays:
                raise ValueError(
                    f"{path} lacks sensitivity required for h-posterior scoring"
                )
            probability = _posterior_after_nondetection(
                prediction, np.asarray(arrays["estimated_sensitivity"], dtype=float)
            )
            ranking = probability
            ranking_source = "h_posterior_after_standard_nondetection"
        else:
            probability = None
            ranking = np.asarray(arrays.get("ranking_score", prediction), dtype=float)
            ranking_source = "saved_ranking_only"
        if ranking.shape != prediction.shape:
            raise ValueError(f"{path} has a ranking score with the wrong shape")
        candidates = measured & ~observed
        if not np.all(np.isfinite(ranking[candidates])) or (
            probability is not None and not np.all(np.isfinite(probability[candidates]))
        ):
            raise ValueError(
                "Undefined/nonfinite natural-candidate score; this commonly means "
                f"p=e=1 in {path}"
            )
        return {
            "cells": cells,
            "targets": targets,
            "truth": reference,
            "candidates": candidates,
            "ranking": ranking,
            "probability": probability,
            "semantics": semantics_value,
            "ranking_source": ranking_source,
        }

    def assert_panel_copy(canonical, duplicate, *, context: str) -> None:
        for field in ("cells", "targets", "truth", "candidates"):
            if not np.array_equal(canonical[field], duplicate[field]):
                raise ValueError(
                    f"Natural panel-seed copies disagree in {field}: {context}"
                )
        if not np.array_equal(
            canonical["ranking"], duplicate["ranking"], equal_nan=True
        ):
            raise ValueError(f"Natural panel-seed rankings disagree: {context}")
        canonical_probability = canonical["probability"]
        duplicate_probability = duplicate["probability"]
        if (canonical_probability is None) != (duplicate_probability is None):
            raise ValueError(f"Natural panel-seed probability semantics disagree: {context}")
        if canonical_probability is not None and not np.array_equal(
            canonical_probability, duplicate_probability, equal_nan=True
        ):
            raise ValueError(f"Natural panel-seed probabilities disagree: {context}")

    for (dataset, repetition), contexts in sorted(expected_contexts.items()):
        for model in sorted(semantics):
            recorded = {
                (panel_seed, outer_fold)
                for (current_dataset, current_repetition, current_model,
                     panel_seed, outer_fold) in prediction_paths
                if (current_dataset, current_repetition, current_model)
                == (dataset, repetition, model)
            }
            if recorded != contexts:
                raise ValueError(
                    "Natural-recovery prediction files are incomplete for "
                    f"{dataset}, data repetition={repetition}, model={model}; "
                    f"missing={sorted(contexts.difference(recorded))}, "
                    f"extra={sorted(recorded.difference(contexts))}"
                )

    per_target_rows = []
    budget_target_rows = []
    audit_rows = []
    reference_candidates: dict[tuple[str, int, str], tuple[np.ndarray, np.ndarray]] = {}
    seen_models_by_target: dict[tuple[str, int, str], set[str]] = {}
    work = [
        (dataset, repetition, model)
        for dataset, repetition in sorted(expected_contexts)
        for model in sorted(semantics)
    ]
    for work_index, (dataset, repetition, model) in enumerate(work, start=1):
        contexts = expected_contexts[(dataset, repetition)]
        panel_seeds = sorted({panel_seed for panel_seed, _ in contexts})
        canonical_seed = panel_seeds[0]
        folds = sorted({outer_fold for panel_seed, outer_fold in contexts
                        if panel_seed == canonical_seed})
        for panel_seed in panel_seeds[1:]:
            other_folds = sorted({outer_fold for current_seed, outer_fold in contexts
                                  if current_seed == panel_seed})
            if other_folds != folds:
                raise ValueError(
                    f"Natural panel seeds have different fold sets: {dataset}, "
                    f"data repetition={repetition}, seed={panel_seed}"
                )
        print(
            f"[V3 Projection {work_index}/{len(work)}] verify panel copies and "
            f"evaluate {dataset}, data repetition={repetition}, model={model}",
            flush=True,
        )
        cells_by_target: dict[str, list[np.ndarray]] = {}
        truth_by_target: dict[str, list[np.ndarray]] = {}
        ranking_by_target: dict[str, list[np.ndarray]] = {}
        probability_by_target: dict[str, list[np.ndarray]] = {}
        ranking_source = None
        duplicate_candidates = 0
        for outer_fold in folds:
            canonical = load_candidate_arrays(
                prediction_paths[(
                    dataset, repetition, model, canonical_seed, outer_fold
                )],
                model,
            )
            if ranking_source is None:
                ranking_source = str(canonical["ranking_source"])
            elif ranking_source != canonical["ranking_source"]:
                raise ValueError(f"Natural folds mix ranking sources for {model}")
            for panel_seed in panel_seeds[1:]:
                duplicate = load_candidate_arrays(
                    prediction_paths[(
                        dataset, repetition, model, panel_seed, outer_fold
                    )],
                    model,
                )
                assert_panel_copy(
                    canonical,
                    duplicate,
                    context=(f"{dataset}, data repetition={repetition}, model={model}, "
                             f"fold={outer_fold}, panel seed={panel_seed}"),
                )
                duplicate_candidates += int(np.asarray(
                    canonical["candidates"], dtype=bool
                ).sum())
            for column, target in enumerate(canonical["targets"]):
                selected = np.asarray(canonical["candidates"], dtype=bool)[:, column]
                cells_by_target.setdefault(str(target), []).append(
                    np.asarray(canonical["cells"])[selected]
                )
                truth_by_target.setdefault(str(target), []).append(
                    np.asarray(canonical["truth"], dtype=bool)[selected, column]
                )
                ranking_by_target.setdefault(str(target), []).append(
                    np.asarray(canonical["ranking"], dtype=float)[selected, column]
                )
                if canonical["probability"] is not None:
                    probability_by_target.setdefault(str(target), []).append(
                        np.asarray(canonical["probability"], dtype=float)[selected, column]
                    )

        model_candidate_count = 0
        for target in sorted(cells_by_target):
            cells = np.concatenate(cells_by_target[target]).astype(str)
            truth = np.concatenate(truth_by_target[target]).astype(bool)
            ranking = np.concatenate(ranking_by_target[target]).astype(float)
            probability = (
                np.concatenate(probability_by_target[target]).astype(float)
                if target in probability_by_target else None
            )
            identity_order = np.argsort(cells, kind="stable")
            cells = cells[identity_order]
            truth = truth[identity_order]
            ranking = ranking[identity_order]
            if probability is not None:
                probability = probability[identity_order]
            if len(np.unique(cells)) != len(cells):
                raise ValueError(
                    "A natural candidate appears in multiple outer folds in the "
                    f"canonical panel seed: {dataset}, data repetition={repetition}, "
                    f"model={model}, target={target}"
                )
            reference_key = (dataset, repetition, target)
            if reference_key not in reference_candidates:
                reference_candidates[reference_key] = (cells.copy(), truth.copy())
            else:
                reference_cells, reference_truth = reference_candidates[reference_key]
                if not (
                    np.array_equal(cells, reference_cells)
                    and np.array_equal(truth, reference_truth)
                ):
                    raise ValueError(
                        "Natural candidate identities/reference labels differ across models "
                        f"for {dataset}, data repetition={repetition}, target={target}"
                    )
            seen_models_by_target.setdefault(reference_key, set()).add(model)
            model_candidate_count += len(truth)
            scores = _metric_vector(truth, probability, ranking)
            per_target_rows.append({
                "dataset": dataset,
                "repetition": repetition,
                "model": model,
                "target": target,
                "evaluation_scope": "hidden_candidate",
                "mechanism": "natural",
                "condition_roles": "natural_recovery",
                "probability_semantics": semantics[model],
                "candidate_probability": (
                    "h_after_standard_nondetection" if probability is not None else "NA"
                ),
                "ranking_source": ranking_source,
                "candidate_count": len(truth),
                "amplification_confirmed_positive_count": int(truth.sum()),
                "amplification_reference_negative_count": int((~truth).sum()),
                "source_panel_seed_count": len(panel_seeds),
                "duplicate_candidates_collapsed": len(truth) * (len(panel_seeds) - 1),
                **scores,
            })
            ranked_order = np.lexsort((cells, -ranking))
            ranked_truth = truth[ranked_order]
            for budget in budgets:
                selected_count = (
                    max(1, int(np.ceil(budget * len(ranked_truth))))
                    if len(ranked_truth) else 0
                )
                budget_target_rows.append({
                    "dataset": dataset,
                    "repetition": repetition,
                    "model": model,
                    "target": target,
                    "budget_fraction": budget,
                    "selected_candidates": selected_count,
                    "candidate_count": len(ranked_truth),
                    "amplification_confirmed_positive_count": int(truth.sum()),
                    "recovered_positive_count": int(
                        ranked_truth[:selected_count].sum()
                    ),
                    "ranking_source": ranking_source,
                })
        audit_rows.append({
            "dataset": dataset,
            "data_repetition": repetition,
            "model": model,
            "probability_semantics": semantics[model],
            "canonical_panel_seed": canonical_seed,
            "source_panel_seed_count": len(panel_seeds),
            "outer_fold_count": len(folds),
            "prediction_file_count": len(panel_seeds) * len(folds),
            "candidate_count": model_candidate_count,
            "duplicate_candidates_collapsed": duplicate_candidates,
            "candidate_identity_check": "matched_across_models",
            "panel_seed_copy_check": "identical",
        })

    for key, models in seen_models_by_target.items():
        if models != set(semantics):
            raise ValueError(
                f"Natural candidate target {key} is missing models: "
                f"{sorted(set(semantics).difference(models))}"
            )

    print(
        "[V3 Projection] candidate identities matched; aggregate macro and "
        "budget metrics",
        flush=True,
    )

    per_target = pd.DataFrame(per_target_rows)
    group_keys = [
        "dataset",
        "repetition",
        "model",
        "evaluation_scope",
        "mechanism",
        "condition_roles",
        "probability_semantics",
        "candidate_probability",
        "ranking_source",
    ]
    metric_names = (
        "auprc",
        "auroc",
        "brier",
        "log_loss",
        "predicted_prevalence",
        "reference_prevalence",
    )
    summary_rows = []
    for key, frame in per_target.groupby(group_keys, dropna=False, observed=True):
        prefix = dict(zip(group_keys, key if isinstance(key, tuple) else (key,)))
        row = {
            **prefix,
            "n_targets": int(len(frame)),
            "n_nonempty_targets": int((frame["candidate_count"] > 0).sum()),
            "n_evaluable_targets": int(np.isfinite(
                pd.to_numeric(frame["auprc"], errors="coerce")
            ).sum()),
            "candidate_count": int(frame["candidate_count"].sum()),
            "amplification_confirmed_positive_count": int(
                frame["amplification_confirmed_positive_count"].sum()
            ),
            "amplification_reference_negative_count": int(
                frame["amplification_reference_negative_count"].sum()
            ),
            "scanned_prediction_files": len(prediction_paths),
            "source_panel_seed_count": int(frame["source_panel_seed_count"].max()),
            "duplicate_candidates_collapsed": int(
                frame["duplicate_candidates_collapsed"].sum()
            ),
        }
        for metric in metric_names:
            row[f"macro_{metric}"] = float(
                pd.to_numeric(frame[metric], errors="coerce").mean()
            )
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)

    budget_per_target = pd.DataFrame(budget_target_rows)
    budget_keys = ["dataset", "repetition", "model", "budget_fraction"]
    budget_summary = (
        budget_per_target.groupby(budget_keys, dropna=False, observed=True)[
            [
                "selected_candidates",
                "candidate_count",
                "amplification_confirmed_positive_count",
                "recovered_positive_count",
            ]
        ]
        .sum()
        .reset_index()
    )
    ranking_sources = (
        budget_per_target.groupby(budget_keys, dropna=False, observed=True)["ranking_source"]
        .agg(lambda values: ";".join(sorted(set(values.astype(str)))))
        .reset_index()
    )
    budget_summary = budget_summary.merge(
        ranking_sources, on=budget_keys, how="left", validate="one_to_one"
    )
    # The legacy column is retained so existing plot/report code can consume
    # the V3 table.  augment_projection_budget_metrics replaces it with the
    # target-macro value below.
    budget_summary["amplification_confirmed_recall"] = np.nan
    budget_summary, budget_per_target = augment_projection_budget_metrics(
        budget_summary, per_target=budget_per_target
    )
    print(
        f"[V3 Projection] complete: {len(summary)} model summary row(s), "
        f"{len(per_target)} model-target row(s), "
        f"{len(budget_summary)} budget row(s)",
        flush=True,
    )
    return {
        "summary": summary,
        "per_target": per_target,
        "budget": budget_summary,
        "budget_per_target": budget_per_target,
        "audit": pd.DataFrame(audit_rows),
    }


def plot_projection_natural_comparison(
    table: pd.DataFrame,
    *,
    models: Sequence[str] = KEY_MODELS,
    title: str | None = None,
) -> tuple[Figure, np.ndarray]:
    """Show candidate-pool ranking and h-probability accuracy."""
    frame = _as_frame(
        table,
        name="Projection natural candidate metrics",
        required=("model", "macro_auprc", "macro_auroc", "macro_brier", "macro_log_loss"),
    )
    order = [model for model in models if model in set(frame["model"].astype(str))]
    if not order:
        raise ValueError("Projection natural metrics contain none of the requested models")
    specifications = (
        ("macro_auprc", "Candidate AUPRC ↑"),
        ("macro_log_loss", "Candidate h-log loss ↓"),
        ("macro_auroc", "Candidate AUROC ↑"),
        ("macro_brier", "Candidate h-Brier ↓"),
    )
    figure, axes = plt.subplots(2, 2, figsize=(12.0, 7.8), constrained_layout=True)
    for axis, (metric, label) in zip(axes.flat, specifications):
        values = (
            frame.assign(**{metric: _numeric(frame, metric)})
            .groupby("model", observed=True)[metric]
            .mean()
            .reindex(order)
        )
        y = np.arange(len(order))
        finite = np.isfinite(values.to_numpy(dtype=float))
        axis.scatter(
            values.to_numpy(dtype=float)[finite],
            y[finite],
            s=58,
            color=[_model_color(order[index], index) for index in np.flatnonzero(finite)],
            zorder=3,
        )
        for index in np.flatnonzero(~finite):
            axis.text(0.5, index, "NA", transform=axis.get_yaxis_transform(),
                      ha="center", va="center", color="#777777")
        axis.set_yticks(y, order)
        axis.invert_yaxis()
        axis.set_xlabel(label)
        axis.grid(axis="x", color="#E6E6E6", linewidth=0.7)
        axis.spines[["top", "right", "left"]].set_visible(False)
    figure.suptitle(
        title
        or "Projection-TAGs standard-negative candidates — amplification-reference evaluation"
    )
    return figure, axes


def _coverage_label(prefix: str, requested: float, actual: float) -> str:
    requested_label = f"{100 * requested:g}%"
    if np.isclose(requested, actual, atol=5e-4):
        return f"{prefix}{requested_label}"
    return f"{prefix}{requested_label}→{100 * actual:.1f}%"


def _facet_values(frame: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    if "sharing_strength" not in frame or frame["sharing_strength"].isna().all():
        return [("all", pd.Series(True, index=frame.index, dtype=bool))]
    numeric = pd.to_numeric(frame["sharing_strength"], errors="coerce")
    values = sorted(numeric.dropna().unique())
    return [
        (f"sharing={value:g}", np.isclose(numeric, value)) for value in values
    ]


def prepare_measurement_funky_table(
    per_repetition: pd.DataFrame,
    *,
    measurement_protocol: Mapping[str, object],
    projection_natural: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Create the fixed scenario × metric table used by both V3 summaries."""
    required = (
        "model",
        "condition_roles",
        "evaluation_scope",
        "mechanism",
        "gene_requested_coverage",
        "gene_coverage",
        "target_requested_coverage",
        "target_coverage",
        "positive_retention",
    )
    frame = _as_frame(per_repetition, name="per_repetition.csv", required=required)
    if frame.empty:
        raise ValueError("per_repetition.csv is empty")
    for column in (
        "gene_requested_coverage",
        "gene_coverage",
        "target_requested_coverage",
        "target_coverage",
        "positive_retention",
    ):
        frame[column] = _numeric(frame, column)
    anchor_gene = float(measurement_protocol["anchor_gene_coverage"])
    anchor_target = float(measurement_protocol["anchor_target_coverage"])
    anchor_retention = float(measurement_protocol["anchor_retention"])
    metric_columns = [specification[0] for specification in FUNKY_METRICS]
    missing_metrics = set(metric_columns).difference(frame)
    if missing_metrics:
        raise ValueError(
            f"per_repetition.csv lacks funky-summary metrics: {sorted(missing_metrics)}"
        )
    records = []
    condition_records = []

    def append_conditions(source: pd.DataFrame, family: str, coordinates, order_start: int):
        for offset, (condition, condition_label, selected) in enumerate(coordinates):
            for metric_order, (metric, metric_label, glyph, higher) in enumerate(
                FUNKY_METRICS
            ):
                condition_records.append({
                    "family": family,
                    "family_order": order_start,
                    "condition": condition,
                    "condition_label": condition_label,
                    "condition_order": order_start + offset,
                    "metric": metric,
                    "metric_label": metric_label,
                    "metric_order": metric_order,
                    "glyph": glyph,
                    "higher_is_better": higher,
                })
            if selected.empty:
                continue
            for facet, facet_mask in _facet_values(selected):
                current = selected.loc[facet_mask]
                if current.empty:
                    continue
                for model, model_rows in current.groupby("model", observed=True):
                    semantics_values = tuple(sorted(set(
                        model_rows.get(
                            "probability_semantics", pd.Series("unknown", index=model_rows.index)
                        ).dropna().astype(str)
                    )))
                    semantics = semantics_values[0] if len(semantics_values) == 1 else "mixed_records"
                    for metric_order, (metric, metric_label, glyph, higher) in enumerate(
                        FUNKY_METRICS
                    ):
                        value = float(pd.to_numeric(model_rows[metric], errors="coerce").mean())
                        if metric in {"macro_brier", "macro_log_loss"} and not semantics.startswith(
                            REFERENCE_PROBABILITY_PREFIX
                        ):
                            value = np.nan
                        records.append({
                            "facet": facet,
                            "model": str(model),
                            "family": family,
                            "family_order": order_start,
                            "condition": condition,
                            "condition_label": condition_label,
                            "condition_order": order_start + offset,
                            "metric": metric,
                            "metric_label": metric_label,
                            "metric_order": metric_order,
                            "glyph": glyph,
                            "higher_is_better": higher,
                            "value": value,
                            "probability_semantics": semantics,
                            "n_rows_averaged": int(len(model_rows)),
                        })

    primary = frame.loc[
        frame["evaluation_scope"].astype(str).eq("native_reference")
        & frame["mechanism"].astype(str).eq("assay_target_sar")
    ].copy()
    full = primary.loc[_roles(primary, "full_control")]
    append_conditions(
        primary,
        "Full control",
        [("full", "G100/T100/P100", full)],
        0,
    )

    retention = primary.loc[
        _roles(primary, "retention_curve")
        & np.isclose(primary["gene_requested_coverage"], anchor_gene)
        & np.isclose(primary["target_requested_coverage"], anchor_target)
    ]
    retention_conditions = []
    declared_retentions = tuple(map(float, measurement_protocol["retentions"]))
    for value in sorted(declared_retentions, reverse=True):
        retention_conditions.append((
            f"retention_{value:.12g}",
            f"P{100 * value:g}%",
            retention.loc[np.isclose(retention["positive_retention"], value)],
        ))
    append_conditions(retention, "Positive retention", retention_conditions, 10)

    coverage = primary.loc[
        _roles(primary, "coverage_heatmap")
        & np.isclose(primary["positive_retention"], anchor_retention)
    ]
    joint_conditions = []
    declared_gene_coverages = tuple(map(float, measurement_protocol["gene_coverages"]))
    declared_target_coverages = tuple(map(float, measurement_protocol["target_coverages"]))
    for gene_requested in sorted(declared_gene_coverages, reverse=True):
        for target_requested in sorted(declared_target_coverages, reverse=True):
            selected = coverage.loc[
                np.isclose(coverage["gene_requested_coverage"], gene_requested)
                & np.isclose(
                    coverage["target_requested_coverage"], target_requested
                )
            ]
            gene_actual = (
                float(selected["gene_coverage"].mean())
                if not selected.empty else gene_requested
            )
            target_actual = (
                float(selected["target_coverage"].mean())
                if not selected.empty else target_requested
            )
            joint_conditions.append((
                f"coverage_g{gene_requested:.12g}_t{target_requested:.12g}",
                (
                    f"{_coverage_label('G', gene_requested, gene_actual)} / "
                    f"{_coverage_label('T', target_requested, target_actual)}"
                ),
                selected,
            ))
    append_conditions(
        coverage, "Gene × target coverage", joint_conditions, 20
    )

    scar = frame.loc[
        frame["evaluation_scope"].astype(str).eq("native_reference")
        & frame["mechanism"].astype(str).eq("scar")
        & _roles(frame, "matched_uniform_control")
    ]
    if bool(measurement_protocol.get("include_matched_uniform", True)):
        append_conditions(
            scar,
            "Matched SCAR",
            [("matched_scar", f"SCAR P{100 * anchor_retention:g}%", scar)],
            40,
        )

    if projection_natural is not None and not projection_natural.empty:
        natural = _as_frame(
            projection_natural,
            name="Projection natural V3 metrics",
            required=(
                "model",
                "evaluation_scope",
                "mechanism",
                "condition_roles",
                *metric_columns,
            ),
        )
        natural = natural.loc[
            natural["evaluation_scope"].astype(str).eq("hidden_candidate")
            & natural["mechanism"].astype(str).eq("natural")
            & _roles(natural, "natural_recovery")
        ]
        append_conditions(
            natural,
            "Amplification-confirmed recovery",
            [("natural_candidates", "standard D=0", natural)],
            50,
        )

    observed = pd.DataFrame(records)
    if observed.empty:
        raise ValueError("No declared measurement scenarios were available for the funky summary")
    catalog = (
        pd.DataFrame(condition_records)
        .drop_duplicates(["condition", "metric"])
        .sort_values(["family_order", "condition_order", "metric_order"], kind="stable")
        .reset_index(drop=True)
    )
    facets = [facet for facet, _ in _facet_values(frame)]
    present_models = set(observed["model"].astype(str))
    models = [*FULL_MODEL_ORDER, *sorted(present_models.difference(FULL_MODEL_ORDER))]
    skeleton = pd.MultiIndex.from_product(
        [facets, models, catalog.index],
        names=["facet", "model", "catalog_index"],
    ).to_frame(index=False)
    skeleton = skeleton.merge(
        catalog.rename_axis("catalog_index").reset_index(),
        on="catalog_index",
        how="left",
        validate="many_to_one",
    ).drop(columns="catalog_index")
    values = observed[[
        "facet", "model", "condition", "metric", "value",
        "probability_semantics", "n_rows_averaged",
    ]]
    if values.duplicated(["facet", "model", "condition", "metric"]).any():
        raise ValueError("Funky summary has duplicate model × condition × metric rows")
    result = skeleton.merge(
        values,
        on=["facet", "model", "condition", "metric"],
        how="left",
        validate="one_to_one",
    )
    result["probability_semantics"] = result["probability_semantics"].fillna(
        "not_available"
    )
    result["n_rows_averaged"] = result["n_rows_averaged"].fillna(0).astype(int)
    # One common metric range per dataset/facet makes key and full figures use
    # identical normalization and avoids magnifying tiny within-cell deltas.
    eligible = ~result["model"].isin(EXTRA_INFORMATION_MODELS)
    result["desirability"] = np.nan
    for (facet, metric), indices in result.groupby(
        ["facet", "metric"], dropna=False, observed=True
    ).groups.items():
        indices = pd.Index(indices)
        reference = result.loc[indices[eligible.loc[indices] & np.isfinite(
            pd.to_numeric(result.loc[indices, "value"], errors="coerce")
        )], "value"].astype(float)
        if reference.empty:
            continue
        lower, upper = float(reference.min()), float(reference.max())
        values = pd.to_numeric(result.loc[indices, "value"], errors="coerce")
        if np.isclose(lower, upper):
            normalized = pd.Series(0.5, index=indices, dtype=float)
        else:
            normalized = (values - lower) / (upper - lower)
        higher = bool(result.loc[indices, "higher_is_better"].iloc[0])
        if not higher:
            normalized = 1.0 - normalized
        result.loc[indices, "desirability"] = normalized.clip(0, 1)
    return result.sort_values(
        ["facet", "family_order", "condition_order", "metric_order", "model"],
        kind="stable",
    ).reset_index(drop=True)


def _row_group(model: str) -> str:
    if model in {"Logistic", "MIRT", "Joint"}:
        return "Non-PU baselines"
    if model in {"PU", "PU-MIRT", "PU-Joint"}:
        return "Gene2Wire PU family"
    if model.startswith("Reference"):
        return "Reference-label controls"
    if model.startswith("RF-"):
        return "Random-forest controls"
    if model.startswith("Qiao"):
        return "Qiao baselines"
    if model in {"GenEML-adapted", "Inductive-PU-MC", "SAR-PU"}:
        return "External PU comparators"
    return "Other"


def plot_measurement_funky_summary(
    table: pd.DataFrame,
    *,
    mode: str,
    key_models: Sequence[str] = KEY_MODELS,
    title: str | None = None,
) -> tuple[Figure, np.ndarray]:
    """Render a dependency-free funkyheatmap-style benchmark summary.

    Circles encode ranking metrics and bars encode proper losses.  Glyph size,
    fill and length all point in the favorable direction.  Exact raw values are
    overlaid in the key view; the full view keeps glyphs legible.
    """
    frame = _as_frame(
        table,
        name="funky summary table",
        required=(
            "facet",
            "model",
            "family",
            "family_order",
            "condition",
            "condition_label",
            "condition_order",
            "metric",
            "metric_label",
            "metric_order",
            "glyph",
            "higher_is_better",
            "value",
            "desirability",
        ),
    )
    models = _models_for_mode(frame, mode, key_models)
    frame = frame.loc[frame["model"].isin(models)].copy()
    facets = list(dict.fromkeys(frame["facet"].astype(str)))
    columns = (
        frame[[
            "family",
            "family_order",
            "condition",
            "condition_label",
            "condition_order",
            "metric",
            "metric_label",
            "metric_order",
            "glyph",
        ]]
        .drop_duplicates()
        .sort_values(["family_order", "condition_order", "metric_order"], kind="stable")
        .reset_index(drop=True)
    )
    column_keys = list(zip(columns["condition"], columns["metric"]))
    width = max(12.0, 2.8 + 0.52 * len(columns))
    per_axis_height = 1.8 + 0.43 * len(models)
    figure, axes = plt.subplots(
        len(facets),
        1,
        figsize=(width, per_axis_height * len(facets)),
        squeeze=False,
        constrained_layout=True,
    )
    axes = axes[:, 0]
    cmap_circle = plt.get_cmap("Blues")
    cmap_bar = plt.get_cmap("Purples")
    family_colors = ("#F3F6FA", "#FFF7ED", "#F3FAF7", "#F8F3FA", "#FFF4F4", "#F4F4F4")
    for facet_index, (axis, facet) in enumerate(zip(axes, facets)):
        current = frame.loc[frame["facet"].astype(str).eq(facet)]
        lookup = current.set_index(["model", "condition", "metric"], drop=False)
        # Alternate softly colored family bands and label each block once.
        for family_index, (family, family_columns) in enumerate(
            columns.groupby("family", sort=False, observed=True)
        ):
            positions = family_columns.index.to_numpy(dtype=int)
            start, end = positions.min() - 0.5, positions.max() + 0.5
            axis.axvspan(start, end, color=family_colors[family_index % len(family_colors)], zorder=0)
            axis.text(
                (start + end) / 2,
                1.13,
                family,
                transform=axis.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="bold",
            )
            if start > -0.5:
                axis.axvline(start, color="white", linewidth=2.0, zorder=1)
        for condition, condition_columns in columns.groupby(
            "condition", sort=False, observed=True
        ):
            positions = condition_columns.index.to_numpy(dtype=int)
            start, end = positions.min() - 0.5, positions.max() + 0.5
            axis.axvline(start, color="#CCCCCC", linewidth=0.65, zorder=1)
            axis.text(
                (start + end) / 2,
                1.035,
                str(condition_columns["condition_label"].iloc[0]),
                transform=axis.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=7.5,
                rotation=35,
            )
        for row, model in enumerate(models):
            for column, (condition, metric) in enumerate(column_keys):
                key = (model, condition, metric)
                if key not in lookup.index:
                    axis.add_patch(Rectangle(
                        (column - 0.34, row - 0.27), 0.68, 0.54,
                        facecolor="#E6E6E6", edgecolor="white", linewidth=0.5,
                        zorder=2,
                    ))
                    axis.text(column, row, "NA", ha="center", va="center",
                              fontsize=5.5, color="#777777", zorder=4)
                    continue
                value_row = lookup.loc[key]
                if isinstance(value_row, pd.DataFrame):
                    value_row = value_row.iloc[0]
                value = float(value_row["value"]) if np.isfinite(value_row["value"]) else np.nan
                desirability = (
                    float(value_row["desirability"])
                    if np.isfinite(value_row["desirability"])
                    else np.nan
                )
                if not np.isfinite(value) or not np.isfinite(desirability):
                    axis.add_patch(Rectangle(
                        (column - 0.34, row - 0.27), 0.68, 0.54,
                        facecolor="#E6E6E6", edgecolor="white", linewidth=0.5,
                        zorder=2,
                    ))
                    axis.text(column, row, "NA", ha="center", va="center",
                              fontsize=5.5, color="#777777", zorder=4)
                    continue
                if value_row["glyph"] == "circle":
                    axis.scatter(
                        [column],
                        [row],
                        s=26 + 250 * desirability,
                        color=[cmap_circle(0.22 + 0.75 * desirability)],
                        edgecolor="white",
                        linewidth=0.5,
                        zorder=3,
                    )
                else:
                    axis.add_patch(Rectangle(
                        (column - 0.36, row - 0.20), 0.72, 0.40,
                        facecolor="#ECECEC", edgecolor="white", linewidth=0.4,
                        zorder=2,
                    ))
                    axis.add_patch(Rectangle(
                        (column - 0.36, row - 0.20), 0.72 * desirability, 0.40,
                        facecolor=cmap_bar(0.24 + 0.70 * desirability),
                        edgecolor="none", zorder=3,
                    ))
                if mode == "key":
                    axis.text(
                        column,
                        row,
                        f"{value:.3f}",
                        ha="center",
                        va="center",
                        fontsize=5.2,
                        color="black" if desirability < 0.73 else "white",
                        zorder=4,
                    )
        axis.set_xlim(-0.5, len(columns) - 0.5)
        axis.set_ylim(len(models) - 0.5, -0.5)
        axis.set_yticks(np.arange(len(models)), models)
        axis.set_xticks(
            np.arange(len(columns)),
            [
                f"{label}{'↑' if higher else '↓'}"
                for label, higher in zip(
                    columns["metric_label"],
                    columns.merge(
                        frame[["metric", "higher_is_better"]].drop_duplicates(),
                        on="metric", how="left"
                    )["higher_is_better"],
                )
            ],
            rotation=90,
            fontsize=6.5,
        )
        axis.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
        axis.tick_params(axis="y", length=0)
        axis.set_title(facet if facet != "all" else "", loc="left", pad=58, fontsize=10)
        for spine in axis.spines.values():
            spine.set_visible(False)
        # Row-family separators preserve model identities without sorting by score.
        previous = None
        for row, model in enumerate(models):
            current_group = _row_group(model)
            if previous is not None and current_group != previous:
                axis.axhline(row - 0.5, color="#BDBDBD", linewidth=0.8, zorder=4)
            previous = current_group
    figure.suptitle(
        title or f"Measurement degradation — {mode} models (funkyheatmap-style)",
        fontsize=13,
    )
    return figure, axes


## Positive-retention degradation curves

Key and full-model views pair native-reference Macro-AUPRC with reference-probability Macro log loss. Both x axes run from 100% retention at the left toward 0% at the right; observed/mixed probability models show ranking metrics but are omitted from proper loss.


In [ ]:
retention_figure_paths = {}
for label, artifacts in all_artifacts.items():
    per_repetition = artifacts.tables.get('per_repetition', pd.DataFrame())
    retention = _role_rows(per_repetition, 'retention_curve')
    required = {'positive_retention', 'macro_auprc', 'macro_log_loss', 'model'}
    if retention.empty or not required.issubset(retention.columns):
        raise RuntimeError(f'{label}: retention metrics are missing or incomplete.')
    for column in ('positive_retention', 'macro_auprc', 'macro_log_loss'):
        retention[column] = pd.to_numeric(retention[column], errors='coerce')
    retention = retention.loc[
        np.isfinite(retention['positive_retention'])
        & np.isfinite(retention['macro_auprc'])
    ]
    if retention.empty:
        raise RuntimeError(f'{label}: no finite native-reference retention rows.')
    for group_label, group in _plot_groups(retention):
        for mode in ('key', 'full'):
            options = ({'key_models': KEY_MEASUREMENT_MODELS} if mode == 'key' else {})
            figure, axes = plot_degradation_pair(
                group,
                x_col='positive_retention',
                x_label='Positive retention',
                mode=mode,
                title=f'{label} ({group_label}) — {mode} models',
                **options,
            )
            suffix = f'{group_label}_retention_auprc_log_loss_{mode}_v3'
            retention_figure_paths[(label, group_label, mode)] = _save_show(
                figure, label, suffix)
display(retention_figure_paths)


## Gene-coverage and target-coverage degradation curves

For both dimensions, key and full-model views pair Macro-AUPRC with reference-probability Macro log loss. The saved manifest supplies the anchor conditions, realized integer coverage is plotted, and both x axes run from 100% at the left toward 0% at the right.


In [ ]:
coverage_figure_paths = {}
for label, artifacts in all_artifacts.items():
    coverage = _role_rows(
        artifacts.tables.get('per_repetition', pd.DataFrame()), 'coverage_heatmap')
    required = {
        'gene_requested_coverage', 'gene_coverage',
        'target_requested_coverage', 'target_coverage',
        'positive_retention', 'macro_auprc', 'macro_log_loss', 'model',
    }
    if coverage.empty or not required.issubset(coverage.columns):
        raise RuntimeError(f'{label}: coverage-curve coordinates are missing or incomplete.')
    for column in required.difference({'model'}):
        coverage[column] = pd.to_numeric(coverage[column], errors='coerce')
    saved_protocol = getattr(artifacts, 'manifest', {}).get('measurement_protocol', {})
    saved_anchor_gene_coverage = float(saved_protocol.get(
        'anchor_gene_coverage', ANCHOR_GENE_COVERAGE))
    saved_anchor_target_coverage = float(saved_protocol.get(
        'anchor_target_coverage', ANCHOR_TARGET_COVERAGE))
    saved_anchor_retention = float(saved_protocol.get(
        'anchor_retention', ANCHOR_POSITIVE_RETENTION))
    common = coverage.loc[
        np.isclose(coverage['positive_retention'], saved_anchor_retention)
        & np.isfinite(coverage['macro_auprc'])
    ].copy()
    curve_specs = (
        (
            'gene',
            common.loc[np.isclose(
                common['target_requested_coverage'], saved_anchor_target_coverage)],
            'gene_coverage',
            f'Gene coverage (target anchor={saved_anchor_target_coverage:g})',
        ),
        (
            'target',
            common.loc[np.isclose(
                common['gene_requested_coverage'], saved_anchor_gene_coverage)],
            'target_coverage',
            f'Target coverage (gene anchor={saved_anchor_gene_coverage:g})',
        ),
    )
    for dimension, curve, coverage_col, subtitle in curve_specs:
        curve = curve.loc[np.isfinite(curve[coverage_col])]
        if curve.empty:
            raise RuntimeError(
                f'{label}: no finite {dimension}-coverage rows at the saved anchors.')
        for group_label, group in _plot_groups(curve):
            for mode in ('key', 'full'):
                options = ({'key_models': KEY_MEASUREMENT_MODELS} if mode == 'key' else {})
                figure, axes = plot_degradation_pair(
                    group,
                    x_col=coverage_col,
                    x_label=subtitle,
                    mode=mode,
                    title=(f'{label} ({group_label}) — {dimension} degradation; '
                           f'retention={saved_anchor_retention:g}; {mode} models'),
                    **options,
                )
                suffix = f'{group_label}_{dimension}_coverage_auprc_log_loss_{mode}_v3'
                coverage_figure_paths[(label, group_label, dimension, mode)] = _save_show(
                    figure, label, suffix)
display(coverage_figure_paths)


## Joint gene-coverage × target-coverage heatmap

One external baseline is selected using development validation loss over the whole grid,
then frozen. PU-MIRT is a PU-Joint structural ablation, so neither PU-MIRT nor PU-Joint
is eligible to be the fixed-baseline candidate. Positive color values mean
`Brier(fixed external baseline) - Brier(PU-Joint) > 0` and therefore favor PU-Joint.
A baseline must have converged results for the complete development grid; none is
selected cell-by-cell or from test metrics.


In [ ]:
heatmap_figure_paths = {}
for label, artifacts in all_artifacts.items():
    selection = artifacts.tables.get('heatmap_baseline_selection', pd.DataFrame())
    if selection.empty or not {'model', 'selected'}.issubset(selection.columns):
        raise RuntimeError(f'{label}: fixed development-selected heatmap baseline was not recorded.')
    selected_flag = selection['selected'].astype(str).str.lower().isin(('true', '1'))
    selected_baseline = selection.loc[selected_flag, 'model']
    if len(selected_baseline) != 1:
        raise RuntimeError(f'{label}: expected exactly one fixed heatmap baseline.')
    baseline_model = str(selected_baseline.iloc[0])
    heatmap = _role_rows(
        artifacts.tables.get('per_repetition', pd.DataFrame()), 'coverage_heatmap')
    for column in ('gene_coverage', 'target_coverage', 'macro_brier'):
        heatmap[column] = pd.to_numeric(heatmap[column], errors='coerce')
    heatmap = heatmap.loc[np.isfinite(heatmap['macro_brier'])]
    if heatmap.empty:
        raise RuntimeError(f'{label}: no native-reference coverage-heatmap metrics were recorded.')
    for group_label, group in _plot_groups(heatmap):
        figure, axis = plot_gene_target_brier_heatmap(
            group, baseline_model=baseline_model, comparison_model='PU-Joint',
            title=f'{label} ({group_label}) — fixed {baseline_model} contrast')
        # All degradation coordinates run from 100% at left toward 0% at right.
        if axis.get_xlim()[0] < axis.get_xlim()[1]:
            axis.invert_xaxis()
        suffix = f'{group_label}_gene_target_brier_heatmap_v3'
        heatmap_figure_paths[(label, group_label)] = _save_show(figure, label, suffix)
display(heatmap_figure_paths)


## Funkyheatmap-style model scorecards

Following the [funkyheatmap visual grammar](https://funkyheatmap.github.io/funkyheatmap/), exactly two scorecards are generated with repository-native Matplotlib: key models and all fitted models. This keeps results-only execution independent of a new R/Python package installation. Rows are models; grouped columns are declared masking mechanism × degree × metric. Circles encode AUPRC/AUROC and bars encode Brier/log loss; larger and darker always means better. Proper losses are `NA` when a model does not output reference probabilities. Projection-TAGs additionally includes amplification-confirmed recovery among standard-negative candidates.


In [ ]:
funky_figure_paths = {}
projection_v3_tables = {}
for label, artifacts in all_artifacts.items():
    projection_natural = None
    if label == 'Projection-TAGs':
        projection_v3_tables[label] = derive_projection_natural_metrics(artifacts)
        projection_natural = projection_v3_tables[label]['summary']
    saved_protocol = getattr(artifacts, 'manifest', {}).get('measurement_protocol', {})
    if not saved_protocol:
        raise RuntimeError(f'{label}: manifest lacks measurement_protocol.')
    scorecard = prepare_measurement_funky_table(
        artifacts.tables.get('per_repetition', pd.DataFrame()),
        measurement_protocol=saved_protocol,
        projection_natural=projection_natural,
    )
    scorecard_path = _figure_path(label, 'funkyheatmap_scorecard_values_v3').with_suffix('.csv')
    scorecard.to_csv(scorecard_path, index=False)
    display(scorecard_path)
    for mode in ('key', 'full'):
        options = ({'key_models': KEY_MEASUREMENT_MODELS} if mode == 'key' else {})
        figure, axes = plot_measurement_funky_summary(
            scorecard,
            mode=mode,
            title=f'{label} — {mode} models (funkyheatmap-style)',
            **options,
        )
        funky_figure_paths[(label, mode)] = _save_show(
            figure, label, f'funkyheatmap_{mode}_v3')
display(funky_figure_paths)


## Essential metrics, selected parameters and diagnostics

The visible report includes exact panel/support audits, important-model metrics,
PU-MIRT/PU-Joint/new-comparator selected settings, candidate counts, detector/propensity
checks, convergence/retry/boundary evidence and every failure count. The bounded
PU-Joint trace exposes rank screening, penalty refinement, total shrinkage,
residual/shared ratio, and optimizer/initializer messages. Complete tuning, per-target,
per-group, reliability, checkpoint and prediction records remain exported; no failed
grid cell is silently omitted.


In [ ]:
def _bounded_table(frame, limit=60):
    if SHOW_FULL_DIAGNOSTICS or len(frame) <= limit:
        return frame
    print(f'Showing {limit}/{len(frame)} rows; the complete table is in the export directory.')
    return frame.head(limit)


def _balanced_model_table(frame, limit=60):
    if SHOW_FULL_DIAGNOSTICS or 'model' not in frame or len(frame) <= limit:
        return frame
    models = tuple(dict.fromkeys(frame['model'].astype(str)))
    per_model = max(1, limit // len(models))
    result = pd.concat([
        frame.loc[frame['model'].astype(str).eq(model)].head(per_model)
        for model in models
    ], ignore_index=True)
    print(f'Showing a model-balanced {len(result)}/{len(frame)} rows; '
          'the complete table is in the export directory.')
    return result


def display_measurement_audits(artifacts, label):
    print(f'\n{label} — measurement-design and comparator audit tables')
    for table_name in (
        'measurement_gene_panel_conditions',
        'measurement_gene_panel_assays',
        'measurement_gene_panel_pairs',
        'measurement_gene_panel_multiplicity',
        'measurement_gene_panel_membership',
        'measurement_target_panel_conditions',
        'measurement_target_panel_assays',
        'measurement_target_panel_pairs',
        'measurement_target_panel_multiplicity',
        'measurement_target_panel_membership',
        'measurement_support',
        'measurement_support_assays',
        'measurement_support_pairs',
        'measurement_gene_target_support',
        'measurement_unsupported_gene_target_pairs',
        'measurement_scenarios',
        'observation_diagnostics',
        'heatmap_baseline_selection',
        'heatmap_contrasts',
        'panel_stability',
        'projection_budget_recall',
        'projection_budget_recall_per_target',
    ):
        frame = artifacts.tables.get(table_name, pd.DataFrame())
        if not frame.empty:
            print(f'{table_name}: {len(frame)} rows')
            display(_bounded_table(frame))

    metrics = artifacts.tables.get('metrics', pd.DataFrame())
    if not metrics.empty and 'model' in metrics:
        important = metrics.loc[metrics['model'].isin(KEY_MEASUREMENT_MODELS)].copy()
        columns = [column for column in (
            'dataset', 'sharing_strength', 'condition_roles', 'mechanism',
            'gene_coverage', 'target_coverage', 'positive_retention',
            'evaluation_scope', 'model', 'macro_auprc', 'macro_log_loss',
            'macro_brier', 'macro_predicted_prevalence', 'macro_reference_prevalence',
        ) if column in important]
        print('Important-model metrics (complete fold/repetition rows remain in metrics.csv):')
        display(_balanced_model_table(important.loc[:, columns]))

    selected = artifacts.tables.get('selected', pd.DataFrame())
    if not selected.empty and 'model' in selected:
        important = selected.loc[selected['model'].isin(KEY_MEASUREMENT_MODELS)].copy()
        parameter_hints = (
            'rank', 'l2', 'lambda', 'penalty', 'propensity', 'exposure',
            'factor', 'iteration', 'converged', 'retry', 'objective',
            'structure', 'validation', 'trial', 'budget', 'maxiter',
            'tolerance', 'message', 'status', 'shrinkage', 'ratio',
            'initializer', 'start',
        )
        identity = (
            'dataset', 'sharing_strength', 'condition_roles', 'mechanism',
            'gene_coverage', 'target_coverage', 'positive_retention',
            'repetition', 'outer_fold', 'model',
        )
        columns = [column for column in important if (
            column in identity or any(hint in column.lower() for hint in parameter_hints)
        )]
        print('Selected parameters for PU / structured PU / new comparators:')
        display(_balanced_model_table(important.loc[:, columns]))

    tuning = artifacts.tables.get('tuning', pd.DataFrame())
    if not tuning.empty and 'model' in tuning:
        important = tuning.loc[tuning['model'].isin(KEY_MEASUREMENT_MODELS)].copy()
        groups = [column for column in (
            'model', 'condition_roles', 'mechanism', 'stage'
        ) if column in important]
        if groups:
            important['validation_loss'] = pd.to_numeric(
                important.get('validation_loss'), errors='coerce')
            summary = important.groupby(groups, dropna=False, observed=True).agg(
                recorded_candidate_trials=('model', 'size'),
                best_recorded_validation_loss=('validation_loss', 'min'),
            ).reset_index()
            print('Recorded tuning trials for important models:')
            display(_balanced_model_table(summary))

        joint_trace = important.loc[
            important['model'].eq('PU-Joint')
            & important.get(
                'stage', pd.Series('', index=important.index)
            ).isin(('rank', 'penalty'))
        ].copy()
        if not joint_trace.empty:
            trace_columns = [column for column in (
                'dataset', 'sharing_strength', 'condition_roles', 'mechanism',
                'gene_coverage', 'target_coverage', 'positive_retention',
                'repetition', 'outer_fold', 'model', 'stage', 'index', 'rank',
                'shared_l2', 'residual_l2', 'total_shrinkage',
                'residual_shared_ratio', 'validation_loss', 'converged',
                'iterations', 'optimizer_message',
            ) if column in joint_trace]
            print(
                'PU-Joint adaptive rank/penalty trace '
                '(bounded view; complete rows remain in tuning.csv):'
            )
            display(_bounded_table(joint_trace.loc[:, trace_columns], limit=60))

    detection = artifacts.tables.get('detection', pd.DataFrame())
    if not detection.empty:
        values = [column for column in (
            'detection_log_loss', 'detection_brier', 'detection_mean',
            'detection_observed_rate', 'sensitivity_mae', 'sensitivity_rmse',
        ) if column in detection]
        groups = [column for column in (
            'mechanism', 'positive_retention', 'gene_coverage', 'target_coverage',
        ) if column in detection]
        if values:
            summary = detection.groupby(groups, dropna=False, observed=True)[values].mean().reset_index()
            print('Detection / propensity diagnostics:')
            display(_bounded_table(summary))

    convergence_rows = []
    for table_name, flag in (('selected', 'final_converged'), ('tuning', 'converged')):
        frame = artifacts.tables.get(table_name, pd.DataFrame())
        if not frame.empty and 'model' in frame:
            values = frame.get(flag, pd.Series(index=frame.index, dtype=object))
            if table_name == 'selected' and 'converged' in frame:
                values = values.combine_first(frame['converged'])
            values = values.fillna('not_recorded').astype(str).str.lower()
            for (model, status), count in values.groupby([frame['model'], values]).size().items():
                convergence_rows.append({
                    'table': table_name, 'model': model,
                    'convergence_status': status, 'records': int(count),
                })
    if convergence_rows:
        print('Final-fit and candidate convergence counts:')
        display(_balanced_model_table(pd.DataFrame(convergence_rows)))

    failures = artifacts.tables.get('failures')
    if failures is None:
        print('Failure status was not recorded.')
    elif failures.empty:
        print('Recorded failed units: 0.')
    else:
        print(f'Recorded failed units: {len(failures)}; no failed condition is dropped.')
        display(_bounded_table(failures, limit=20))


## Display bounded shared and measurement-specific reports


In [ ]:
for label, artifacts in all_artifacts.items():
    display_measurement_audits(artifacts, label)
    display_diagnostics(artifacts, label=label, full=SHOW_FULL_DIAGNOSTICS)
print('Figure PDFs:', FIGURE_DIR)
print('Read-only source exports:', {label: str(artifacts.export_dir) for label, artifacts in all_artifacts.items()})
print('No V3 model fitting or checkpoint writes are permitted.')
